##### Copyright 2019 The TensorFlow Authors.

# Text generation with an RNN

Цель работы - познакомиться с одним из подходов к генерации текста при помощи рекурентной нейронной сети.
Задачи:
1. Изучить блокнот.
2. Для одного из датасетов из файла "Ссылки на датсеты" выполнить генерацию текста на 100 и 1000 символов.
3. Выполнить генерацию текста при различном количестве эпох (например, 5, 30, 60).
4. Выполнить геренацию текста при различных функциях потерь и оптимизации.
5. `При наличии времени и желания оценить качество генерации текста.`

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/text/tutorials/text_generation"><img src="https://www.tensorflow.org/images/tf_logo_32px.png" />View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/text/blob/master/docs/tutorials/text_generation.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/text/blob/master/docs/tutorials/text_generation.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/text/docs/tutorials/text_generation.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>
</table>

This tutorial demonstrates how to generate text using a character-based RNN. You will work with a dataset of Shakespeare's writing from Andrej Karpathy's [The Unreasonable Effectiveness of Recurrent Neural Networks](http://karpathy.github.io/2015/05/21/rnn-effectiveness/). Given a sequence of characters from this data ("Shakespear"), train a model to predict the next character in the sequence ("e"). Longer sequences of text can be generated by calling the model repeatedly.

Note: Enable GPU acceleration to execute this notebook faster. In Colab: *Runtime > Change runtime type > Hardware accelerator > GPU*.

This tutorial includes runnable code implemented using [tf.keras](https://www.tensorflow.org/guide/keras/sequential_model) and [eager execution](https://www.tensorflow.org/guide/eager). The following is the sample output when the model in this tutorial trained for 30 epochs, and started with the prompt "Q":

<pre>
QUEENE:
I had thought thou hadst a Roman; for the oracle,
Thus by All bids the man against the word,
Which are so weak of care, by old care done;
Your children were in your holy love,
And the precipitation through the bleeding throne.

BISHOP OF ELY:
Marry, and will, my lord, to weep in such a one were prettiest;
Yet now I was adopted heir
Of the world's lamentable day,
To watch the next way with his father with his face?

ESCALUS:
The cause why then we are all resolved more sons.

VOLUMNIA:
O, no, no, no, no, no, no, no, no, no, no, no, no, no, no, no, no, no, no, no, no, it is no sin it should be dead,
And love and pale as any will to that word.

QUEEN ELIZABETH:
But how long have I heard the soul for this world,
And show his hands of life be proved to stand.

PETRUCHIO:
I say he look'd on, if I must be content
To stay him from the fatal of our country's bliss.
His lordship pluck'd from this sentence then for prey,
And then let us twain, being the moon,
were she such a case as fills m
</pre>

While some of the sentences are grammatical, most do not make sense. The model has not learned the meaning of words, but consider:

* The model is character-based. When training started, the model did not know how to spell an English word, or that words were even a unit of text.

* The structure of the output resembles a play—blocks of text generally begin with a speaker name, in all capital letters similar to the dataset.

* As demonstrated below, the model is trained on small batches of text (100 characters each), and is still able to generate a longer sequence of text with coherent structure.

## Setup

### Import TensorFlow and other libraries

In [60]:
import tensorflow as tf

import numpy as np
import os
import time


import os, sys
import time
import kagglehub
import zipfile
import io
import pandas as pd
import chardet
import csv
import subprocess

### Download the Shakespeare dataset

Change the following line to run this code on your own data.

In [61]:
path_to_file = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')

### Read the data

First, look in the text:

In [62]:
# Read, then decode for py2 compat.
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
# length of text is the number of characters in it
print(f'Length of text: {len(text)} characters')

Length of text: 1115394 characters


In [63]:
# ----------------------------------------------------------------------
# Загрузка данных
# ----------------------------------------------------------------------
# Загрузка датасета с помощью kagglehub
dataset_path = kagglehub.dataset_download("trainingdatapro/generated-e-mail-spam")
print("Путь к файлам датасета:", dataset_path)

# Функция для загрузки текста из zip-архива
def load_text_from_zip(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_file:
        # Получаем имя первого CSV-файла в zip-архиве
        text_file_name = None
        for name in zip_file.namelist():
            if name.endswith(".csv"):
                text_file_name = name
                break
        if not text_file_name:
            raise Exception("CSV-файл не найден в zip-архиве")

        with zip_file.open(text_file_name, 'r') as f:
            df = pd.read_csv(f, encoding = 'latin1')
            # Объединяем все текстовые столбцы в одну строку
            text_columns = df.select_dtypes(include='object').columns.tolist()
            text = " ".join(df[text_columns].astype(str).fillna('').values.flatten())
    return text

# Функция для загрузки текста из локального текстового файла
def load_text_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    return text

def load_text_from_csv(csv_file_path, delimiter=','):
        try:
            with open(csv_file_path, 'rb') as f:
                raw_data = f.read()
                result = chardet.detect(raw_data)
                encoding = result['encoding']
            with open(csv_file_path, 'r', encoding=encoding, errors='ignore') as f:
                lines = [line.strip() for line in f]

            temp_file = "temp.csv"
            with open(temp_file, 'w', encoding=encoding, errors='ignore') as f:
                f.write('\n'.join(lines))

            try:
                # Попробуем прочитать с предположением, что нет заголовков
                df = pd.read_csv(temp_file, encoding=encoding, on_bad_lines='skip',
                                 delimiter=delimiter, header=None,
                                 skipinitialspace=True)
            except pd.errors.ParserError as e:
                print(f"Ошибка парсинга CSV (header=None): {e}")
                # Попробуем прочитать с предположением, что есть заголовки
                df = pd.read_csv(temp_file, encoding=encoding, on_bad_lines='skip',
                                 delimiter=delimiter, skipinitialspace=True)
            os.remove(temp_file)
        except Exception as e:
            print(f"Ошибка загрузки CSV с кодировкой '{encoding}' или 'latin1': {e}")
            df = pd.read_csv(csv_file_path, encoding='latin1', on_bad_lines='skip', delimiter=delimiter)
        # Объединяем все текстовые столбцы в одну строку
        text_columns = df.select_dtypes(include='object').columns.tolist()
        text = " ".join(df[text_columns].astype(str).fillna('').values.flatten())
        return text


# Проверяем, является ли датасет zip-архивом или директорией
text = None
try:
    if os.path.isdir(dataset_path):
        # Если директория, ищем .csv или .txt файлы внутри
        csv_file_path = None
        text_file_path = None
        for file in os.listdir(dataset_path):
            if file.endswith(".csv"):
                csv_file_path = os.path.join(dataset_path, file)
                break
            elif file.endswith(".txt"):
                text_file_path = os.path.join(dataset_path,file)
                break
        if csv_file_path:
             text = load_text_from_csv(csv_file_path) # default delimiter is ","
        elif text_file_path:
            text = load_text_from_file(text_file_path)
        else:
            raise Exception("Подходящий файл не найден в директории датасета")
    elif dataset_path.endswith(".zip"):
        text = load_text_from_zip(dataset_path)
    else:
        raise Exception("Датасет не является zip-архивом или директорией")
except Exception as e:
    print(f"Ошибка загрузки датасета: {e}")
    exit()

Путь к файлам датасета: C:\Users\VV\.cache\kagglehub\datasets\trainingdatapro\generated-e-mail-spam\versions\1


In [64]:
# Take a look at the first 250 characters in text
print(text[:250])

Title	Text The Love-Booster Crew" Unbelievable Simple Trick to Help Make Your Life Easier	"Yo! What's up! Do you ever feel like you just can't seem to catch a break? Are you always drowning in work and responsibilities? BOB'S MAGIC is ridiculously si


In [65]:
# The unique characters in the file
vocab = sorted(set(text))
print(f'{len(vocab)} unique characters')

102 unique characters


## Process the text

### Vectorize the text

Before training, you need to convert the strings to a numerical representation. 

The `tf.keras.layers.StringLookup` layer can convert each character into a numeric ID. It just needs the text to be split into tokens first.

In [66]:
example_texts = ['abcdefg', 'xyz']

chars = tf.strings.unicode_split(example_texts, input_encoding='UTF-8')
chars

<tf.RaggedTensor [[b'a', b'b', b'c', b'd', b'e', b'f', b'g'], [b'x', b'y', b'z']]>

Now create the `tf.keras.layers.StringLookup` layer:

In [67]:
ids_from_chars = tf.keras.layers.StringLookup(
    vocabulary=list(vocab), mask_token=None)

It converts from tokens to character IDs:

In [68]:
ids = ids_from_chars(chars)
ids

<tf.RaggedTensor [[58, 59, 60, 61, 62, 63, 64], [81, 82, 83]]>

Since the goal of this tutorial is to generate text, it will also be important to invert this representation and recover human-readable strings from it. For this you can use `tf.keras.layers.StringLookup(..., invert=True)`.  

Note: Here instead of passing the original vocabulary generated with `sorted(set(text))` use the `get_vocabulary()` method of the `tf.keras.layers.StringLookup` layer so that the `[UNK]` tokens is set the same way.

In [69]:
chars_from_ids = tf.keras.layers.StringLookup(
    vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)

This layer recovers the characters from the vectors of IDs, and returns them as a `tf.RaggedTensor` of characters:

In [70]:
chars = chars_from_ids(ids)
chars

<tf.RaggedTensor [[b'a', b'b', b'c', b'd', b'e', b'f', b'g'], [b'x', b'y', b'z']]>

You can `tf.strings.reduce_join` to join the characters back into strings. 

In [71]:
tf.strings.reduce_join(chars, axis=-1).numpy()

array([b'abcdefg', b'xyz'], dtype=object)

In [72]:
def text_from_ids(ids):
  return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

### The prediction task

Given a character, or a sequence of characters, what is the most probable next character? This is the task you're training the model to perform. The input to the model will be a sequence of characters, and you train the model to predict the output—the following character at each time step.

Since RNNs maintain an internal state that depends on the previously seen elements, given all the characters computed until this moment, what is the next character?


### Create training examples and targets

Next divide the text into example sequences. Each input sequence will contain `seq_length` characters from the text.

For each input sequence, the corresponding targets contain the same length of text, except shifted one character to the right.

So break the text into chunks of `seq_length+1`. For example, say `seq_length` is 4 and our text is "Hello". The input sequence would be "Hell", and the target sequence "ello".

To do this first use the `tf.data.Dataset.from_tensor_slices` function to convert the text vector into a stream of character indices.

In [73]:
all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
all_ids

<tf.Tensor: shape=(24053,), dtype=int64, numpy=array([48, 66, 77, ..., 42, 39, 26], shape=(24053,))>

In [74]:
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)

In [75]:
for ids in ids_dataset.take(2):
    print(chars_from_ids(ids).numpy().decode('utf-8'))

T
i


In [76]:
seq_length = 100


The `batch` method lets you easily convert these individual characters to sequences of the desired size.

In [77]:
sequences = ids_dataset.batch(seq_length+1, drop_remainder=True)

for seq in sequences.take(1):
  print(chars_from_ids(seq))

tf.Tensor(
[b'T' b'i' b't' b'l' b'e' b'\t' b'T' b'e' b'x' b't' b' ' b'T' b'h' b'e'
 b' ' b'L' b'o' b'v' b'e' b'-' b'B' b'o' b'o' b's' b't' b'e' b'r' b' '
 b'C' b'r' b'e' b'w' b'"' b' ' b'U' b'n' b'b' b'e' b'l' b'i' b'e' b'v'
 b'a' b'b' b'l' b'e' b' ' b'S' b'i' b'm' b'p' b'l' b'e' b' ' b'T' b'r'
 b'i' b'c' b'k' b' ' b't' b'o' b' ' b'H' b'e' b'l' b'p' b' ' b'M' b'a'
 b'k' b'e' b' ' b'Y' b'o' b'u' b'r' b' ' b'L' b'i' b'f' b'e' b' ' b'E'
 b'a' b's' b'i' b'e' b'r' b'\t' b'"' b'Y' b'o' b'!' b' ' b'W' b'h' b'a'
 b't' b"'" b's'], shape=(101,), dtype=string)


It's easier to see what this is doing if you join the tokens back into strings:

In [78]:
for seq in sequences.take(5):
  print(text_from_ids(seq).numpy())

b'Title\tText The Love-Booster Crew" Unbelievable Simple Trick to Help Make Your Life Easier\t"Yo! What\'s'
b" up! Do you ever feel like you just can't seem to catch a break? Are you always drowning in work and "
b"responsibilities? BOB'S MAGIC is ridiculously simple to use and helps you smash through your to-do li"
b'st in no time. Say goodbye to stress and hello to productivity with this game-changing trick! The BOB'
b'\'S MAGIC Crew" Find your ride or die today and have an epic summer full of unforgettable memories! Th'


For training you'll need a dataset of `(input, label)` pairs. Where `input` and 
`label` are sequences. At each time step the input is the current character and the label is the next character. 

Here's a function that takes a sequence as input, duplicates, and shifts it to align the input and label for each timestep:

In [79]:
def split_input_target(sequence):
    input_text = sequence[:-1] #Hell
    target_text = sequence[1:] #ello
    return input_text, target_text

In [80]:
split_input_target(list("Tensorflow"))

(['T', 'e', 'n', 's', 'o', 'r', 'f', 'l', 'o'],
 ['e', 'n', 's', 'o', 'r', 'f', 'l', 'o', 'w'])

In [81]:
dataset = sequences.map(split_input_target)

In [82]:
for input_example, target_example in dataset.take(1):
    print("Input :", text_from_ids(input_example).numpy())
    print("Target:", text_from_ids(target_example).numpy())

Input : b'Title\tText The Love-Booster Crew" Unbelievable Simple Trick to Help Make Your Life Easier\t"Yo! What\''
Target: b'itle\tText The Love-Booster Crew" Unbelievable Simple Trick to Help Make Your Life Easier\t"Yo! What\'s'


### Create training batches

You used `tf.data` to split the text into manageable sequences. But before feeding this data into the model, you need to shuffle the data and pack it into batches.

In [83]:
# Batch size
BATCH_SIZE = 64

# Buffer size to shuffle the dataset
# (TF data is designed to work with possibly infinite sequences,
# so it doesn't attempt to shuffle the entire sequence in memory. Instead,
# it maintains a buffer in which it shuffles elements).
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE))

dataset

<_PrefetchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>

## Build The Model

This section defines the model as a `keras.Model` subclass (For details see [Making new Layers and Models via subclassing](https://www.tensorflow.org/guide/keras/custom_layers_and_models)). 

This model has three layers:

* `tf.keras.layers.Embedding`: The input layer. A trainable lookup table that will map each character-ID to a vector with `embedding_dim` dimensions;
* `tf.keras.layers.GRU`: A type of RNN with size `units=rnn_units` (You can also use an LSTM layer here.)
* `tf.keras.layers.Dense`: The output layer, with `vocab_size` outputs. It outputs one logit for each character in the vocabulary. These are the log-likelihood of each character according to the model.

In [84]:
# Length of the vocabulary in StringLookup Layer
vocab_size = len(ids_from_chars.get_vocabulary())

# The embedding dimension
embedding_dim = 256

# Number of RNN units
rnn_units = 1024

In [85]:
class MyModel(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__()
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(rnn_units,
                                   return_sequences=True,
                                   return_state=True)
    self.dense = tf.keras.layers.Dense(vocab_size)

  def call(self, inputs, states=None, return_state=False, training=False):
    x = inputs
    x = self.embedding(x, training=training)
    if states is None:
      states = self.gru.get_initial_state(tf.shape(x)[0])
    x, states = self.gru(x, initial_state=states, training=training)
    x = self.dense(x, training=training)

    if return_state:
      return x, states
    else:
      return x

In [86]:
model = MyModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units)

For each character the model looks up the embedding, runs the GRU one timestep with the embedding as input, and applies the dense layer to generate logits predicting the log-likelihood of the next character:

![A drawing of the data passing through the model](images/text_generation_training.png)

Note: For training you could use a `keras.Sequential` model here. To  generate text later you'll need to manage the RNN's internal state. It's simpler to include the state input and output options upfront, than it is to rearrange the model architecture later. For more details see the [Keras RNN guide](https://www.tensorflow.org/guide/keras/rnn#rnn_state_reuse).

## Try the model

Now run the model to see that it behaves as expected.

First check the shape of the output:

In [87]:
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")

(64, 100, 103) # (batch_size, sequence_length, vocab_size)


In the above example the sequence length of the input is `100` but the model can be run on inputs of any length:

In [88]:
model.summary()

Model: "my_model_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (64, 100, 256)         │        26,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ ((64, 100, 1024), (64, │     3,938,304 │
│                                 │ 1024))                 │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (64, 100, 103)         │       105,575 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,070,247 (15.53 MB)

 Trainable params: 4,070,247 (15.53 MB)

 Non-trainable params: 0 (0.00 B)

To get actual predictions from the model you need to sample from the output distribution, to get actual character indices. This distribution is defined by the logits over the character vocabulary.

Note: It is important to _sample_ from this distribution as taking the _argmax_ of the distribution can easily get the model stuck in a loop.

Try it for the first example in the batch:

In [89]:
sampled_indices = tf.random.categorical(example_batch_predictions[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices, axis=-1).numpy()

This gives us, at each timestep, a prediction of the next character index:

In [90]:
sampled_indices

array([100,  33,  98,  16,  53,  96,  91,   2,  69,  66,  26,   6,   5,
         7,  61,  94,  37,  47,  49,  52,  14,   6,  58,  11,  32,  61,
         7,  63,  91,  19,  11,  42,  18,  93,  73,  55,  19,  62,  75,
        40,  12,  43,  72,  26,  37,  17,  62,  45,  36,  82,   2,  86,
        23,  76,  67,  45,  90,  17,  64,  35,  92,  32,   6,  67,  74,
        54,  74,  22,  64,   5,  64,  68, 100,  99,  14,  17,  61,  95,
        91,   5,  43,  32,   4,  72,  74,  15,  99,  78,  77,  17,  73,
        79,  19,  89,  74,  22,  38,  43,  41,  73])

Decode these to see the text predicted by this untrained model:

In [91]:
print("Input:\n", text_from_ids(input_example_batch[0]).numpy())
print()
print("Next Char Predictions:\n", text_from_ids(sampled_indices).numpy())

Input:
 b'ake action now and start your journey towards a debt-free future! The Team at <LINK>" EraseDebt.net '

Next Char Predictions:
 b'\xc3\xbcE\xc3\xb12Y\xc3\xad\xc3\x91 li>$#%d\xc3\xabISUX0$a*Dd%f\xc3\x915*N4\xc3\xa5p[5erL-Oo>I3eQHy \xc2\xa0:sjQ\xc3\x893gG\xc3\x9cD$jqZq9g#gk\xc3\xbc\xc3\xb203d\xc3\xac\xc3\x91#OD"oq1\xc3\xb2ut3pv5\xc3\x84q9JOMp'


## Train the model

At this point the problem can be treated as a standard classification problem. Given the previous RNN state, and the input this time step, predict the class of the next character.

### Attach an optimizer, and a loss function

The standard `tf.keras.losses.sparse_categorical_crossentropy` loss function works in this case because it is applied across the last dimension of the predictions.

Because your model returns logits, you need to set the `from_logits` flag.


In [92]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)

In [93]:
example_batch_mean_loss = loss(target_example_batch, example_batch_predictions)
print("Prediction shape: ", example_batch_predictions.shape, " # (batch_size, sequence_length, vocab_size)")
print("Mean loss:        ", example_batch_mean_loss)

Prediction shape:  (64, 100, 103)  # (batch_size, sequence_length, vocab_size)
Mean loss:         tf.Tensor(4.634197, shape=(), dtype=float32)


A newly initialized model shouldn't be too sure of itself, the output logits should all have similar magnitudes. To confirm this you can check that the exponential of the mean loss is approximately equal to the vocabulary size. A much higher loss means the model is sure of its wrong answers, and is badly initialized:

In [94]:
tf.exp(example_batch_mean_loss).numpy()

np.float32(102.945244)

Configure the training procedure using the `tf.keras.Model.compile` method. Use `tf.keras.optimizers.Adam` with default arguments and the loss function.

In [95]:
model.compile(optimizer='adam', loss=loss)

### Configure checkpoints

Use a `tf.keras.callbacks.ModelCheckpoint` to ensure that checkpoints are saved during training:

In [96]:
# Directory where the checkpoints will be saved
checkpoint_dir = './training_checkpoints'
# Name of the checkpoint files
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix,
    save_weights_only=True)

### Execute the training

To keep training time reasonable, use 10 epochs to train the model. In Colab, set the runtime to GPU for faster training.

In [97]:
EPOCHS = 50

In [98]:
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - loss: 4.5540   
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 622ms/step - loss: 6.0136
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 685ms/step - loss: 3.9135
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step - loss: 3.9625   
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 638ms/step - loss: 3.8253
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 628ms/step - loss: 3.6005
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - loss: 3.3033   
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 624ms/step - loss: 3.1616
Epoch 9/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 634ms/step - loss: 3.0961
Epoch 10/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - loss: 3.0223   
Epoch 11/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 627ms/step - loss: 2.9199
Epoch 12/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 691ms/step - loss: 2.8546
Epoch 13/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - loss: 2.7821   
Epoch 14/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 622ms/step - loss: 2.7377
Epoch 15/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 629ms/step - loss: 2.6793
Epoch 16/50
3/3 ━━━

## Generate text

The simplest way to generate text with this model is to run it in a loop, and keep track of the model's internal state as you execute it.

![To generate text the model's output is fed back to the input](images/text_generation_sampling.png)

Each time you call the model you pass in some text and an internal state. The model returns a prediction for the next character and its new state. Pass the prediction and state back in to continue generating text.


The following makes a single step prediction:

In [99]:
class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature = temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    # Create a mask to prevent "[UNK]" from being generated.
    skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        # Put a -inf at each bad index.
        values=[-float('inf')]*len(skip_ids),
        indices=skip_ids,
        # Match the shape to the vocabulary
        dense_shape=[len(ids_from_chars.get_vocabulary())])
    self.prediction_mask = tf.sparse.to_dense(sparse_mask)

  @tf.function
  def generate_one_step(self, inputs, states=None):
    # Convert strings to token IDs.
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Run the model.
    # predicted_logits.shape is [batch, char, next_char_logits]
    predicted_logits, states = self.model(inputs=input_ids, states=states,
                                          return_state=True)
    # Only use the last prediction.
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    # Apply the prediction mask: prevent "[UNK]" from being generated.
    predicted_logits = predicted_logits + self.prediction_mask

    # Sample the output logits to generate token IDs.
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    # Convert from token ids to characters
    predicted_chars = self.chars_from_ids(predicted_ids)

    # Return the characters and model state.
    return predicted_chars, states

In [100]:
one_step_model = OneStep(model, chars_from_ids, ids_from_chars)

Run it in a loop to generate some text. Looking at the generated text, you'll see the model knows when to capitalize, make paragraphs and imitates a Shakespeare-like writing vocabulary. With the small number of training epochs, it has not yet learned to form coherent sentences.

In [101]:
start = time.time()
states = None
next_char = tf.constant(['ROMEO:'])
result = [next_char]

for n in range(100):
  next_char, states = one_step_model.generate_one_step(next_char, states=states)
  result.append(next_char)

result = tf.strings.join(result)
end = time.time()
print(result[0].numpy().decode('utf-8'), '\n\n' + '_'*80)
print('\nRun time:', end - start)

ROMEO: Wo you ditn ulnediald le sigs our expertuncite.]" The Mevind the gantherd coment onerss dovez the t 

________________________________________________________________________________

Run time: 0.21585679054260254


The easiest thing you can do to improve the results is to train it for longer (try `EPOCHS = 30`).

You can also experiment with a different start string, try adding another RNN layer to improve the model's accuracy, or adjust the temperature parameter to generate more or less random predictions.

If you want the model to generate text *faster* the easiest thing you can do is batch the text generation. In the example below the model generates 5 outputs in about the same time it took to generate 1 above. 

In [102]:
start = time.time()
states = None
next_char = tf.constant(['ROMEO:', 'ROMEO:', 'ROMEO:', 'ROMEO:', 'ROMEO:'])
result = [next_char]

for n in range(100):
  next_char, states = one_step_model.generate_one_step(next_char, states=states)
  result.append(next_char)

result = tf.strings.join(result)
end = time.time()
print(result, '\n\n' + '_'*80)
print('\nRun time:', end - start)

tf.Tensor(
[b'ROMEO:: Tate Bese the profuties! The Movate life. -waray!" P your mime to inasy you! now AUGDend wa dedper'
 b"ROMEO:. -'s secutt fol yout reevant stermebselifes towarn! We Sistre it moret expecitnance to het Siss sic"
 b'ROMEO:?\xc3\x84\xc3\xbc\xc3\xbcE\xc3\x91Pcrupithe\xef\xa3\xbfAs bangenged of which con\'t mess the reger tomantanig! The Tiam" Don\'t Team" Borove '
 b'ROMEO: just move prof ofper im<P wetteritide tox-At ane nfer Samay! The tak our excamoulliny! Says ot Exbe'
 b'ROMEO:\xc3\x84Jg jon\'t mosh opperoutt at aws find to mank It hight of wire our iqumfay!" Ap offere the con\'t with'], shape=(5,), dtype=string) 

________________________________________________________________________________

Run time: 0.20910906791687012


## Export the generator

This single-step model can easily be [saved and restored](https://www.tensorflow.org/guide/saved_model), allowing you to use it anywhere a `tf.saved_model` is accepted.

In [103]:
tf.saved_model.save(one_step_model, 'one_step')
one_step_reloaded = tf.saved_model.load('one_step')

TypeError: this __dict__ descriptor does not support '_DictWrapper' objects

In [ ]:
states = None
next_char = tf.constant(['ROMEO:'])
result = [next_char]

for n in range(100):
  next_char, states = one_step_reloaded.generate_one_step(next_char, states=states)
  result.append(next_char)

print(tf.strings.join(result)[0].numpy().decode("utf-8"))

NameError: name 'one_step_reloaded' is not defined

## Advanced: Customized Training

The above training procedure is simple, but does not give you much control.
It uses teacher-forcing which prevents bad predictions from being fed back to the model, so the model never learns to recover from mistakes.

So now that you've seen how to run the model manually next you'll implement the training loop. This gives a starting point if, for example, you want to implement _curriculum  learning_ to help stabilize the model's open-loop output.

The most important part of a custom training loop is the train step function.

Use `tf.GradientTape` to track the gradients. You can learn more about this approach by reading the [eager execution guide](https://www.tensorflow.org/guide/eager).

The basic procedure is:

1. Execute the model and calculate the loss under a `tf.GradientTape`.
2. Calculate the updates and apply them to the model using the optimizer.

In [ ]:
class CustomTraining(MyModel):
  @tf.function
  def train_step(self, inputs):
      inputs, labels = inputs
      with tf.GradientTape() as tape:
          predictions = self(inputs, training=True)
          loss = self.loss(labels, predictions)
      grads = tape.gradient(loss, model.trainable_variables)
      self.optimizer.apply_gradients(zip(grads, model.trainable_variables))

      return {'loss': loss}

The above implementation of the `train_step` method follows [Keras' `train_step` conventions](https://www.tensorflow.org/guide/keras/customizing_what_happens_in_fit). This is optional, but it allows you to change the behavior of the train step and still use keras' `Model.compile` and `Model.fit` methods.

In [ ]:
model = CustomTraining(
    vocab_size=len(ids_from_chars.get_vocabulary()),
    embedding_dim=embedding_dim,
    rnn_units=rnn_units)

In [ ]:
model.compile(optimizer = tf.keras.optimizers.Adam(),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

In [ ]:
model.fit(dataset, epochs=1)

3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 207ms/step - loss: 0.0000e+00


Or if you need more control, you can write your own complete custom training loop:

In [ ]:
EPOCHS = 10

mean = tf.metrics.Mean()

for epoch in range(EPOCHS):
    start = time.time()

    mean.reset_states()
    for (batch_n, (inp, target)) in enumerate(dataset):
        logs = model.train_step([inp, target])
        mean.update_state(logs['loss'])

        if batch_n % 50 == 0:
            template = f"Epoch {epoch+1} Batch {batch_n} Loss {logs['loss']:.4f}"
            print(template)

    # saving (checkpoint) the model every 5 epochs
    if (epoch + 1) % 5 == 0:
        model.save_weights(checkpoint_prefix.format(epoch=epoch))

    print()
    print(f'Epoch {epoch+1} Loss: {mean.result().numpy():.4f}')
    print(f'Time taken for 1 epoch {time.time() - start:.2f} sec')
    print("_"*80)

model.save_weights(checkpoint_prefix.format(epoch=epoch))

AttributeError: 'Mean' object has no attribute 'reset_states'